# 🏥 Discharge Summary Agent
**Agentic AI for Clinical Discharge Summaries — Part 1**

This notebook implements a real agent loop that:
- Reads all patient PDFs from a folder
- Plans and re-plans dynamically based on what it finds
- Never fabricates clinical facts
- Flags conflicts, missing data, and pending results
- Emits a full structured trace for every step

In [4]:
!mkdir -p /content/patient_1

In [5]:
!cp "/content/patient_1.pdf" "/content/patient_1/"

## Step 1 — Install Dependencies

In [6]:
!pip install -q pdfplumber pdf2image pytesseract openai pymupdf
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pdfplumber pdf2image pytesseract groq pymupdf
print('Setup complete')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2

## Step 2 — Imports & Configuration

In [7]:
import os
import gc
import json
import re
import time
import pdfplumber
import pytesseract
from groq import Groq
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────────────────────
GROQ_API_KEY   = "sk-xxxxxxxxxxxx"
MODEL          = "llama-3.1-8b-instant"
MAX_STEPS      = 20
PATIENT_FOLDER = "/content/patient_1"
OUTPUT_DIR     = "/content/outputs"
OCR_DPI        = 150

client = Groq(api_key=GROQ_API_KEY)   # ← uses the variable above

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK')

Imports OK


## Step 3 — PDF Ingestion (Multi-Document)

In [8]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Try pdfplumber first (fast, no RAM spike).
    Fall back to OCR page-by-page — frees each image after use
    so a 70-page PDF doesn't OOM Colab.
    """
    fname = os.path.basename(pdf_path)

    # ── Attempt 1: pdfplumber (text-layer PDFs) ──────────────────────────
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages_text = []
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    pages_text.append(t)
            text = "\n".join(pages_text).strip()
        if len(text) > 100:
            print(f"  [pdfplumber] {fname} — {len(text)} chars")
            return text
    except Exception as e:
        print(f"  [pdfplumber] failed on {fname}: {e}")

    # ── Attempt 2: page-by-page OCR (scanned PDFs) ───────────────────────
    # Process ONE page at a time — image is deleted after OCR to free RAM.
    # This prevents the OOM crash that kills Colab on multi-page PDFs.
    try:
        from pdf2image import convert_from_path
        import fitz  # PyMuPDF — used to get page count without loading images

        doc = fitz.open(pdf_path)
        n_pages = doc.page_count
        doc.close()
        print(f"  [OCR] {fname} — {n_pages} pages, processing one at a time...")

        full_text = ""
        for page_num in range(1, n_pages + 1):
            try:
                # Load ONLY this one page as an image
                images = convert_from_path(
                    pdf_path,
                    dpi=OCR_DPI,          # 150 dpi — readable & RAM-safe
                    first_page=page_num,
                    last_page=page_num
                )
                img = images[0]
                page_text = pytesseract.image_to_string(img)
                full_text += f"\n\n=== PAGE {page_num} ===\n{page_text}"

                # Explicitly free the image from memory
                img.close()
                del images, img
                gc.collect()

            except Exception as page_err:
                print(f"  [OCR] page {page_num} failed: {page_err}")
                full_text += f"\n\n=== PAGE {page_num} ===\n[OCR FAILED FOR THIS PAGE]"

        print(f"  [OCR] {fname} done — {len(full_text)} chars")
        return full_text.strip()

    except ImportError:
        # fitz not installed — fall back to safer low-dpi batch
        try:
            from pdf2image import convert_from_path
            print(f"  [OCR-fallback] {fname} — loading all pages at 100dpi")
            images = convert_from_path(pdf_path, dpi=100)
            text = ""
            for i, img in enumerate(images):
                text += f"\n\n=== PAGE {i+1} ===\n"
                text += pytesseract.image_to_string(img)
                img.close()
                gc.collect()
            del images
            gc.collect()
            return text.strip()
        except Exception as e:
            print(f"  [OCR-fallback] failed: {e}")
            return ""

    except Exception as e:
        print(f"  [OCR] failed on {fname}: {e}")
        return ""


def load_patient_documents(folder: str) -> dict:
    """
    Load all PDFs from a patient folder.
    Returns dict: {filename: extracted_text}
    """
    documents = {}
    if not os.path.exists(folder):
        print(f"WARNING: Folder '{folder}' not found.")
        return {}

    pdf_files = [f for f in os.listdir(folder) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"WARNING: No PDFs found in '{folder}'")
        return {}

    print(f"Found {len(pdf_files)} PDFs in {folder}:")
    for fname in sorted(pdf_files):
        fpath = os.path.join(folder, fname)
        print(f"  Loading: {fname}")
        text = extract_text_from_pdf(fpath)
        documents[fname] = text
        gc.collect()   # free after each PDF

    return documents


def build_corpus(documents: dict) -> str:
    """Concatenate all docs into one searchable corpus with source labels."""
    parts = []
    for fname, text in documents.items():
        parts.append(f"\n\n{'='*60}\nSOURCE: {fname}\n{'='*60}\n{text}")
    return "\n".join(parts)


print('PDF ingestion defined (memory-safe OCR)')

PDF ingestion defined (memory-safe OCR)


## Step 4 — LLM Helper

In [9]:
def call_llm(system_prompt: str, user_prompt: str,
             max_tokens: int = 800, retries: int = 3) -> str:
    """
    Wrapper around OpenAI chat completion.
    Retries on failure. Returns empty string if all retries fail.
    """
    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt}
                ],
                max_tokens=max_tokens,
                temperature=0.0      # deterministic — no hallucination drift
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  [LLM] attempt {attempt} failed: {e}")
            if attempt < retries:
                time.sleep(2 ** attempt)   # exponential backoff
    return ""   # all retries exhausted

## Step 5 — Individual Extraction Tools

In [10]:
SYSTEM_EXTRACTOR = """You are a clinical information extraction system.
Rules you must NEVER break:
1. Extract ONLY information explicitly stated in the text.
2. Never infer, guess, or fill in plausible values.
3. If the information is absent, return exactly: MISSING
4. If a result is pending/awaited, return exactly: PENDING — <describe what is pending>
5. Do not add explanations unless asked."""


def extract_field(field_name: str, instruction: str, corpus: str,
                  max_tokens: int = 400) -> str:
    """Generic field extractor — used by all specific extractors below."""
    prompt = f"""{instruction}

Clinical Documents:
---
{corpus[:6000]}
---

{field_name}:"""
    result = call_llm(SYSTEM_EXTRACTOR, prompt, max_tokens=max_tokens)
    if not result:
        return "MISSING — extraction failed (LLM error)"
    return result


# ── Individual tool functions ────────────────────────────────────────────

def tool_extract_demographics(corpus):
    return extract_field(
        "Patient Demographics",
        "Extract patient name, age, gender, MRN/patient ID, and date of birth if present.",
        corpus
    )

def tool_extract_dates(corpus):
    return extract_field(
        "Admission and Discharge Dates",
        "Extract the admission date and discharge date. Return both clearly labelled.",
        corpus
    )

def tool_extract_diagnoses(corpus):
    return extract_field(
        "Diagnoses",
        "Extract ALL diagnoses (principal and secondary). List one per line. "
        "If different documents state different diagnoses, list ALL and note the source document for each.",
        corpus,
        max_tokens=500
    )

def tool_extract_hospital_course(corpus):
    return extract_field(
        "Hospital Course",
        "Summarize the hospital course based ONLY on what is explicitly documented. "
        "Include presenting complaints, key findings, treatment given, and response to treatment.",
        corpus,
        max_tokens=600
    )

def tool_extract_procedures(corpus):
    return extract_field(
        "Procedures and Investigations",
        "Extract all procedures performed and investigations ordered. "
        "For each investigation, note if result is available, pending, or missing.",
        corpus,
        max_tokens=400
    )

def tool_extract_admission_medications(corpus):
    return extract_field(
        "Admission Medications",
        "Extract the medication list at the time of ADMISSION only.",
        corpus
    )

def tool_extract_discharge_medications(corpus):
    return extract_field(
        "Discharge Medications",
        "Extract the medication list at the time of DISCHARGE only.",
        corpus
    )

def tool_extract_allergies(corpus):
    return extract_field(
        "Allergies",
        "Extract all documented allergies and reactions. If none documented, return MISSING.",
        corpus
    )

def tool_extract_followup(corpus):
    return extract_field(
        "Follow-up Instructions",
        "Extract all follow-up instructions, including dates, specialist referrals, and conditions for return.",
        corpus
    )

def tool_extract_pending_results(corpus):
    return extract_field(
        "Pending Results",
        "Extract ALL investigations or results that are explicitly described as pending, awaited, or not yet returned.",
        corpus
    )

def tool_extract_discharge_condition(corpus):
    return extract_field(
        "Condition at Discharge",
        "Extract the patient's documented condition or status at discharge (e.g. stable, improved, critical).",
        corpus
    )

print('Extraction tools defined')

Extraction tools defined


## Step 6 — Safety & Reconciliation Tools

In [11]:
def tool_detect_conflicts(memory: dict, corpus: str) -> list:
    """
    Ask the LLM to find conflicts ACROSS documents.
    Returns list of conflict strings.
    """
    prompt = f"""Review the following clinical documents and identify ANY factual conflicts between them.
Examples: different diagnoses in different notes, different medication doses, conflicting lab values.
If no conflicts found, return: NO CONFLICTS DETECTED
Otherwise list each conflict clearly, noting which documents disagree.

Documents:
{corpus[:6000]}

Conflicts:"""

    result = call_llm(SYSTEM_EXTRACTOR, prompt, max_tokens=500)
    if not result or "NO CONFLICTS" in result.upper():
        return []
    # Split into individual conflict items
    conflicts = [line.strip() for line in result.split("\n") if line.strip()]
    return conflicts


def tool_reconcile_medications(admission_meds: str, discharge_meds: str) -> dict:
    """
    Compare admission vs discharge meds using LLM.
    Flags added, stopped, changed, and undocumented-reason changes.
    """
    if admission_meds.startswith("MISSING") and discharge_meds.startswith("MISSING"):
        return {"error": "Both admission and discharge medication lists are missing."}

    prompt = f"""Compare these two medication lists and identify changes.

ADMISSION MEDICATIONS:
{admission_meds}

DISCHARGE MEDICATIONS:
{discharge_meds}

Return a JSON object with these keys:
- "added": list of medications added at discharge (not in admission)
- "stopped": list of medications present at admission but not at discharge
- "changed": list of medications with dose/frequency changes
- "needs_review": list of any changes with no documented reason (flag these)
- "unchanged": list of medications continued unchanged

If a list is empty, use []. Return ONLY valid JSON."""

    result = call_llm(SYSTEM_EXTRACTOR, prompt, max_tokens=600)
    try:
        # Strip markdown code fences if present
        clean = re.sub(r"```json|```", "", result).strip()
        return json.loads(clean)
    except Exception:
        return {"raw": result, "error": "Could not parse reconciliation as JSON"}


def tool_drug_interaction_check(medications: str) -> str:
    """
    MOCKED external tool — simulates a drug-interaction database lookup.
    In production this would call a real API (e.g. DrugBank, OpenFDA).
    """
    # Mock logic: flag known common interactions for demo purposes
    meds_lower = medications.lower()
    flags = []
    interaction_pairs = [
        ("warfarin",    "aspirin",     "Increased bleeding risk"),
        ("metformin",   "contrast",    "Risk of lactic acidosis with contrast media"),
        ("ofloxacin",   "antacid",     "Reduced ofloxacin absorption with antacids"),
        ("ssri",        "tramadol",    "Serotonin syndrome risk"),
        ("ace inhibitor","potassium",  "Hyperkalemia risk"),
    ]
    for drug_a, drug_b, reason in interaction_pairs:
        if drug_a in meds_lower and drug_b in meds_lower:
            flags.append(f"INTERACTION: {drug_a.upper()} + {drug_b.upper()} — {reason}")

    if not flags:
        return "No known interactions detected (mocked check)."
    return "\n".join(flags)


def tool_flag_for_clinician(memory: dict, issue: str) -> str:
    """Escalation tool — adds a flag to the clinician review list."""
    if "clinician_flags" not in memory:
        memory["clinician_flags"] = []
    if issue not in memory["clinician_flags"]:
        memory["clinician_flags"].append(issue)
    return f"FLAGGED: {issue}"


print('Safety and reconciliation tools defined')

Safety and reconciliation tools defined


## Step 7 — The Agent Brain (Planner)

In [12]:
# All available tools the agent can choose from
AVAILABLE_TOOLS = [
    "extract_demographics",
    "extract_dates",
    "extract_diagnoses",
    "extract_hospital_course",
    "extract_procedures",
    "extract_admission_medications",
    "extract_discharge_medications",
    "extract_allergies",
    "extract_followup",
    "extract_pending_results",
    "extract_discharge_condition",
    "detect_conflicts",
    "reconcile_medications",
    "drug_interaction_check",
    "generate_summary",
    "DONE"
]

PLANNER_SYSTEM = """You are the planning brain of a clinical discharge summary agent.
Your job is to decide what to do NEXT based on current memory state.

Rules:
1. Choose exactly ONE tool from the available list.
2. Reason step by step before deciding.
3. Never skip safety checks (conflicts, reconciliation, drug interactions).
4. Only call generate_summary when ALL fields are attempted and safety checks done.
5. Call DONE only after generate_summary has been called.
6. If both admission AND discharge medications are collected, always run reconcile_medications next.
7. Always run detect_conflicts before generating the summary.
8. Always run drug_interaction_check after discharge medications are extracted."""


def plan_next_action(memory: dict, step: int) -> tuple:
    """
    The agent's planning function.
    Returns (tool_name, reasoning) based on current memory state.
    This is what makes it a real agent — it DECIDES what to do next.
    """
    # Build a concise summary of what we have and what's missing
    memory_state = {}
    for key, val in memory.items():
        if key == "clinician_flags":
            memory_state[key] = f"{len(val)} flags"
        elif key == "reconciliation":
            memory_state[key] = "done" if val else "not done"
        elif val is None:
            memory_state[key] = "NOT YET COLLECTED"
        elif str(val).startswith("MISSING"):
            memory_state[key] = "MISSING"
        else:
            memory_state[key] = "collected"

    prompt = f"""Step {step} of max {MAX_STEPS}.

Current memory state:
{json.dumps(memory_state, indent=2)}

Available tools:
{chr(10).join('- ' + t for t in AVAILABLE_TOOLS)}

Think step by step:
1. What critical information is still missing?
2. Are safety checks (conflicts, reconciliation, drug interactions) done?
3. Is it safe to generate the summary yet?

Respond in this exact format:
REASONING: <your reasoning in 1-3 sentences>
ACTION: <exactly one tool name from the list above>"""

    result = call_llm(PLANNER_SYSTEM, prompt, max_tokens=300)

    # Parse response
    reasoning = "No reasoning provided"
    action = None

    for line in result.split("\n"):
        if line.startswith("REASONING:"):
            reasoning = line.replace("REASONING:", "").strip()
        elif line.startswith("ACTION:"):
            action = line.replace("ACTION:", "").strip()

    # Validate action
    if action not in AVAILABLE_TOOLS:
        # Try to find a partial match
        for tool in AVAILABLE_TOOLS:
            if tool in (result or ""):
                action = tool
                break
        else:
            action = "DONE"  # safe fallback
            reasoning += " [FALLBACK: could not parse action]"

    return action, reasoning


print('Planner defined')

Planner defined


## Step 8 — Tool Executor

In [13]:
def execute_tool(tool_name: str, memory: dict, corpus: str) -> tuple:
    """
    Executes the chosen tool.
    Returns (result, memory_key_updated)
    All failures are caught — agent never crashes.
    """
    try:
        if tool_name == "extract_demographics":
            result = tool_extract_demographics(corpus)
            memory["demographics"] = result
            return result, "demographics"

        elif tool_name == "extract_dates":
            result = tool_extract_dates(corpus)
            memory["dates"] = result
            return result, "dates"

        elif tool_name == "extract_diagnoses":
            result = tool_extract_diagnoses(corpus)
            memory["diagnoses"] = result
            # Auto-flag if multiple sources mentioned (conflict possible)
            if "SOURCE:" in result.upper() or "CONFLICT" in result.upper():
                tool_flag_for_clinician(memory, "Possible diagnosis conflict — multiple sources found")
            return result, "diagnoses"

        elif tool_name == "extract_hospital_course":
            result = tool_extract_hospital_course(corpus)
            memory["hospital_course"] = result
            return result, "hospital_course"

        elif tool_name == "extract_procedures":
            result = tool_extract_procedures(corpus)
            memory["procedures"] = result
            if "PENDING" in result.upper():
                tool_flag_for_clinician(memory, "One or more investigation results are pending")
            return result, "procedures"

        elif tool_name == "extract_admission_medications":
            result = tool_extract_admission_medications(corpus)
            memory["admission_medications"] = result
            return result, "admission_medications"

        elif tool_name == "extract_discharge_medications":
            result = tool_extract_discharge_medications(corpus)
            memory["discharge_medications"] = result
            return result, "discharge_medications"

        elif tool_name == "extract_allergies":
            result = tool_extract_allergies(corpus)
            memory["allergies"] = result
            if result.startswith("MISSING"):
                tool_flag_for_clinician(memory, "Allergy information not documented — clinician must verify")
            return result, "allergies"

        elif tool_name == "extract_followup":
            result = tool_extract_followup(corpus)
            memory["followup"] = result
            return result, "followup"

        elif tool_name == "extract_pending_results":
            result = tool_extract_pending_results(corpus)
            memory["pending_results"] = result
            if not result.startswith("MISSING"):
                tool_flag_for_clinician(memory, f"Pending results require follow-up: {result[:100]}")
            return result, "pending_results"

        elif tool_name == "extract_discharge_condition":
            result = tool_extract_discharge_condition(corpus)
            memory["discharge_condition"] = result
            return result, "discharge_condition"

        elif tool_name == "detect_conflicts":
            conflicts = tool_detect_conflicts(memory, corpus)
            memory["conflicts"] = conflicts
            for c in conflicts:
                tool_flag_for_clinician(memory, f"CONFLICT: {c}")
            result = f"{len(conflicts)} conflict(s) found" if conflicts else "No conflicts detected"
            return result, "conflicts"

        elif tool_name == "reconcile_medications":
            adm  = memory.get("admission_medications", "MISSING")
            disc = memory.get("discharge_medications", "MISSING")
            recon = tool_reconcile_medications(adm, disc)
            memory["reconciliation"] = recon
            # Flag anything needing review
            for item in recon.get("needs_review", []):
                tool_flag_for_clinician(memory, f"Medication change with no documented reason: {item}")
            if recon.get("stopped"):
                tool_flag_for_clinician(memory, f"Medications stopped at discharge: {recon['stopped']}")
            result = json.dumps(recon, indent=2)
            return result, "reconciliation"

        elif tool_name == "drug_interaction_check":
            meds = memory.get("discharge_medications", "")
            result = tool_drug_interaction_check(meds)
            memory["drug_interactions"] = result
            if "INTERACTION" in result:
                tool_flag_for_clinician(memory, f"Drug interaction detected: {result}")
            return result, "drug_interactions"

        elif tool_name == "generate_summary":
            result = generate_discharge_summary(memory)
            memory["summary"] = result
            return result, "summary"

        else:
            return f"Unknown tool: {tool_name}", None

    except Exception as e:
        error_msg = f"Tool '{tool_name}' failed: {str(e)}"
        print(f"  [ERROR] {error_msg}")
        tool_flag_for_clinician(memory, error_msg)
        return error_msg, None


print('Tool executor defined')

Tool executor defined


## Step 9 — Discharge Summary Generator

In [14]:
def generate_discharge_summary(memory: dict) -> str:
    """Formats the agent memory into a structured discharge summary draft."""

    def fmt(val):
        if val is None:
            return "MISSING — not found in source documents"
        return str(val).strip()

    # Medication reconciliation table
    recon = memory.get("reconciliation", {})
    if isinstance(recon, dict) and not recon.get("error"):
        recon_text = (
            f"  Added at discharge  : {recon.get('added', [])}\n"
            f"  Stopped at discharge: {recon.get('stopped', [])}\n"
            f"  Changed             : {recon.get('changed', [])}\n"
            f"  Needs review        : {recon.get('needs_review', [])}\n"
            f"  Unchanged           : {recon.get('unchanged', [])}"
        )
    else:
        recon_text = fmt(recon)

    # Clinician flags
    flags = memory.get("clinician_flags", [])
    flags_text = "\n".join(f"  ⚠️  {f}" for f in flags) if flags else "  None"

    # Conflicts
    conflicts = memory.get("conflicts", [])
    conflicts_text = "\n".join(f"  ❌ {c}" for c in conflicts) if conflicts else "  None detected"

    summary = f"""
{'='*70}
DISCHARGE SUMMARY DRAFT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
⚠️  THIS IS A DRAFT FOR CLINICIAN REVIEW — NOT A FINALIZED DOCUMENT
{'='*70}

PATIENT DEMOGRAPHICS
--------------------
{fmt(memory.get('demographics'))}

ADMISSION & DISCHARGE DATES
---------------------------
{fmt(memory.get('dates'))}

DIAGNOSES
---------
{fmt(memory.get('diagnoses'))}

HOSPITAL COURSE
---------------
{fmt(memory.get('hospital_course'))}

PROCEDURES & INVESTIGATIONS
---------------------------
{fmt(memory.get('procedures'))}

ADMISSION MEDICATIONS
---------------------
{fmt(memory.get('admission_medications'))}

DISCHARGE MEDICATIONS
---------------------
{fmt(memory.get('discharge_medications'))}

MEDICATION RECONCILIATION
-------------------------
{recon_text}

DRUG INTERACTION CHECK
----------------------
{fmt(memory.get('drug_interactions'))}

ALLERGIES
---------
{fmt(memory.get('allergies'))}

FOLLOW-UP INSTRUCTIONS
----------------------
{fmt(memory.get('followup'))}

PENDING RESULTS
---------------
{fmt(memory.get('pending_results'))}

CONDITION AT DISCHARGE
----------------------
{fmt(memory.get('discharge_condition'))}

CONFLICTS DETECTED
------------------
{conflicts_text}

CLINICIAN REVIEW FLAGS  ({'ACTION REQUIRED' if flags else 'None'})
{'='*40}
{flags_text}

{'='*70}
END OF DRAFT — Clinician must review and verify all sections above.
{'='*70}
"""
    return summary


print('Summary generator defined')

Summary generator defined


## Step 10 — The Main Agent Loop

In [15]:
def run_agent(patient_folder: str) -> tuple:
    """
    Main agent loop.

    1. Load all patient PDFs
    2. Plan → Execute → Update memory → Emit trace (repeat)
    3. Stop when DONE or MAX_STEPS reached

    Returns: (memory, trace)
    """
    print(f"\n{'='*60}")
    print(f"AGENT STARTING — Patient folder: {patient_folder}")
    print(f"Max steps: {MAX_STEPS}")
    print(f"{'='*60}\n")

    # ── Load documents ───────────────────────────────────────────────────
    documents = load_patient_documents(patient_folder)
    corpus    = build_corpus(documents)

    if not corpus.strip():
        print("ERROR: No text extracted from documents. Aborting.")
        return {}, []

    print(f"\nTotal corpus size: {len(corpus)} characters across {len(documents)} documents\n")

    # ── Initialize memory ────────────────────────────────────────────────
    memory = {
        # Clinical fields — all start as None (= not yet attempted)
        "demographics":           None,
        "dates":                  None,
        "diagnoses":              None,
        "hospital_course":        None,
        "procedures":             None,
        "admission_medications":  None,
        "discharge_medications":  None,
        "allergies":              None,
        "followup":               None,
        "pending_results":        None,
        "discharge_condition":    None,
        # Safety & meta
        "conflicts":              None,
        "reconciliation":         None,
        "drug_interactions":      None,
        "clinician_flags":        [],
        "summary":                None,
    }

    trace = []   # full structured trace

    # ── Agent loop ───────────────────────────────────────────────────────
    for step in range(1, MAX_STEPS + 1):
        print(f"\n── STEP {step} {'─'*50}")

        # 1. PLAN — agent decides what to do next
        action, reasoning = plan_next_action(memory, step)
        print(f"  REASONING : {reasoning}")
        print(f"  ACTION    : {action}")

        if action == "DONE":
            print("\n  Agent decided: DONE")
            trace.append({
                "step": step,
                "reasoning": reasoning,
                "action": "DONE",
                "input": None,
                "result": "Agent completed successfully",
                "next_decision": "Terminate"
            })
            break

        # 2. EXECUTE — run the chosen tool
        result, memory_key = execute_tool(action, memory, corpus)

        # Truncate long results for trace readability
        result_preview = str(result)[:300] + "..." if len(str(result)) > 300 else str(result)
        print(f"  RESULT    : {result_preview}")

        # 3. EMIT TRACE — structured log of this step
        trace.append({
            "step":          step,
            "reasoning":     reasoning,
            "action":        action,
            "memory_key":    memory_key,
            "result":        result_preview,
            "flags_so_far":  len(memory.get("clinician_flags", []))
        })

    else:
        # Hit step cap
        print(f"\n⚠️  MAX STEPS ({MAX_STEPS}) REACHED — agent stopped.")
        tool_flag_for_clinician(memory, f"Agent hit max step cap ({MAX_STEPS}) — summary may be incomplete")
        if memory.get("summary") is None:
            memory["summary"] = generate_discharge_summary(memory)

    return memory, trace


print('Agent loop defined')

Agent loop defined


## Step 11 — Save Outputs

In [16]:
def save_outputs(memory: dict, trace: list, patient_id: str):
    """Save summary, trace, and flags to output files."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Ensure summary always exists — generate if agent exited early
    if not memory.get('summary'):
        print('  [save] Summary missing — generating now...')
        memory['summary'] = generate_discharge_summary(memory)

    # 1. Discharge summary
    summary_path = os.path.join(OUTPUT_DIR, f"{patient_id}_discharge_summary.txt")
    with open(summary_path, 'w') as f:
        f.write(memory['summary'])
    print(f'Summary saved: {summary_path}')

    # 2. Step trace (JSON)
    trace_path = os.path.join(OUTPUT_DIR, f"{patient_id}_trace.json")
    with open(trace_path, 'w') as f:
        json.dump(trace, f, indent=2)
    print(f'Trace saved:   {trace_path}')

    # 3. Clinician flags
    flags = memory.get('clinician_flags', [])
    flags_path = os.path.join(OUTPUT_DIR, f"{patient_id}_flags.txt")
    with open(flags_path, 'w') as f:
        f.write(f"CLINICIAN FLAGS — {patient_id}\n")
        f.write(f"Generated: {datetime.now()}\n\n")
        if flags:
            for idx, flag in enumerate(flags, 1):
                f.write(f"{idx}. {flag}\n")
        else:
            f.write('No flags.\n')
    print(f'Flags saved:   {flags_path}')

    # 4. Full memory (JSON) — stringify any non-serialisable values
    memory_path = os.path.join(OUTPUT_DIR, f"{patient_id}_memory.json")
    serializable = {}
    for k, v in memory.items():
        if k == 'summary':
            continue
        try:
            json.dumps(v)   # test if serialisable
            serializable[k] = v
        except Exception:
            serializable[k] = str(v)
    with open(memory_path, 'w') as f:
        json.dump(serializable, f, indent=2)
    print(f'Memory saved:  {memory_path}')


print('Output saver defined')

Output saver defined


## Step 12 — Run the Agent 🚀

Upload your patient PDF folder to `/content/patient_1/` and run this cell.

In [17]:
# ── RUN ─────────────────────────────────────────────────────────────────
PATIENT_FOLDER = "/content/patient_1"
PATIENT_ID     = "patient_1"

memory, trace = run_agent(PATIENT_FOLDER)

# Safety net: generate summary if agent exited without producing one
if not memory.get('summary'):
    memory['summary'] = generate_discharge_summary(memory)

save_outputs(memory, trace, PATIENT_ID)

print('\n' + '='*70)
print('FINAL DISCHARGE SUMMARY')
print('='*70)
print(memory['summary'])



AGENT STARTING — Patient folder: /content/patient_1
Max steps: 20

Found 1 PDFs in /content/patient_1:
  Loading: patient_1.pdf
  [OCR] patient_1.pdf — 71 pages, processing one at a time...
  [OCR] patient_1.pdf done — 25100 chars

Total corpus size: 25242 characters across 1 documents


── STEP 1 ──────────────────────────────────────────────────
  REASONING : The critical information missing is demographics, dates, diagnoses, hospital course, procedures, admission medications, discharge medications, allergies, followup, pending results, and discharge condition. Safety checks such as conflicts, reconciliation, and drug interactions have not been done yet. It is not safe to generate the summary yet because some essential information is still missing.
  ACTION    : extract_demographics
  RESULT    : Based on the provided clinical documents, the extracted information is as follows:

1. **Patient Name**: MISSING
2. **Age**: MISSING
3. **Gender**: She (female)
4. **MRN/Patient ID**: MRN: 

## Step 13 — Print Full Step Trace

In [18]:
print("\n" + "="*70)
print("AGENT STEP TRACE")
print("="*70)
for step in trace:
    print(f"\nStep {step['step']}")
    print(f"  Reasoning   : {step['reasoning']}")
    print(f"  Action      : {step['action']}")
    print(f"  Memory key  : {step.get('memory_key', 'N/A')}")
    print(f"  Result      : {step['result'][:150]}")
    print(f"  Flags so far: {step.get('flags_so_far', 0)}")


AGENT STEP TRACE

Step 1
  Reasoning   : The critical information missing is demographics, dates, diagnoses, hospital course, procedures, admission medications, discharge medications, allergies, followup, pending results, and discharge condition. Safety checks such as conflicts, reconciliation, and drug interactions have not been done yet. It is not safe to generate the summary yet because some essential information is still missing.
  Action      : extract_demographics
  Memory key  : demographics
  Result      : Based on the provided clinical documents, the extracted information is as follows:

1. **Patient Name**: MISSING
2. **Age**: MISSING
3. **Gender**: Sh
  Flags so far: 0

Step 2
  Reasoning   : The critical information still missing includes demographics, dates, diagnoses, hospital course, procedures, admission medications, discharge medications, allergies, followup, pending results, and discharge condition. Safety checks such as conflicts, reconciliation, and drug interactio

## Step 14 — Run on Multiple Patients (Batch)

In [19]:
def run_batch(patient_folders: list):
    """
    Run the agent on multiple patients.
    Saves outputs for each independently.
    """
    results = {}
    for folder in patient_folders:
        patient_id = os.path.basename(folder.rstrip("/"))
        print(f"\n{'#'*60}")
        print(f"# PROCESSING: {patient_id}")
        print(f"{'#'*60}")
        try:
            memory, trace = run_agent(folder)
            save_outputs(memory, trace, patient_id)
            results[patient_id] = {
                "status": "success",
                "flags":  len(memory.get("clinician_flags", [])),
                "steps":  len(trace)
            }
        except Exception as e:
            print(f"FATAL ERROR for {patient_id}: {e}")
            results[patient_id] = {"status": "failed", "error": str(e)}

    print("\n" + "="*60)
    print("BATCH SUMMARY")
    print("="*60)
    for pid, res in results.items():
        print(f"  {pid}: {res}")
    return results


# Example — uncomment and set your folders:
# results = run_batch(["/content/patient_1", "/content/patient_2"])